# Classificação de Nível de Renda (Income Level) com Machine Learning
Nesta atividade, utilizamos a base de dados `inflation_cost_of_living_dataset.csv` para uma tarefa de **Classificação**.

**Objetivo:** Prever a classe socioeconômica de um país (a variável `income_level`, ex: High income, Low income) com base em suas métricas numéricas de inflação e índices de custo de vida.

**Requisitos cumpridos:**
- Base de dados com tarefa de Classificação escolhida.
- 3 Classificadores aplicados (Random Forest, SVM e KNN).
- Ajuste de hiperparâmetros (Tuning) para 2 classificadores (Random Forest e SVM) via GridSearchCV.
- Comparação dos modelos via **5-fold Cross Validation**.
- Explicações no código python dos resultados obtidos e especificação do melhor modelo.

In [ ]:
# 1. Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings('ignore')

### 2. Carregamento e Preparação dos Dados
Vamos isolar nossa variável alvo (Target) para classificação e padronizar os dados.

In [ ]:
# Carregando os dados
#Decisão de Projeto: Optamos por selecionar apenas variáveis macroeconômicas na nossa lista de features. Dados geográficos (como latitude e longitude) e de texto (como o nome do país) foram intencionalmente excluídos do treinamento, pois o objetivo é classificar a renda do país puramente pelos seus índices de custo de vida e inflação, evitando viés de localidade.
df = pd.read_csv('/inflation_cost_of_living_dataset.csv')

# A variável alvo da classificação será 'income_level'. As preditoras serão as numéricas.
features = [
    'inflation_annual_pct', 'cost_living_index', 'rent_index',
    'groceries_index', 'restaurants_index', 'purchasing_power_index'
]
target = 'income_level'

# Limpeza de dados nulos nas colunas de interesse
df_limpo = df.dropna(subset=features + [target]).copy()

print(f"Tamanho da base após limpeza: {df_limpo.shape[0]} registros.")
print("\nDistribuição das classes a serem previstas (Income Level):")
print(df_limpo[target].value_counts())

X = df_limpo[features]
y = df_limpo[target]

# Codificando a variável alvo em texto para números numéricos (ex: 0, 1, 2, 3)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Padronizando as variáveis numéricas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Tamanho da base após limpeza: 43 registros.

Distribuição das classes a serem previstas (Income Level):
income_level
High income            26
Upper middle income    10
Lower middle income     6
Low income              1
Name: count, dtype: int64


In [ ]:
df_limpo.head()

,country,country_code_x,region,income_level,capital_city,latitude,longitude,country_code_y,inflation_year,inflation_annual_pct,cost_living_index,rent_index,groceries_index,restaurants_index,purchasing_power_index,bread_price_usd,transport_oneway_usd,transport_monthly_pass_usd,rent_monthly_usd,utilities_monthly_usd
1,Afghanistan,NaN,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,34.5228,69.1761,AFG,2024.0,-6.601186,38.2,8.2,36.8,36.9,48.5,0.88,1.05,12.8,85.0,20.0
5,United Arab Emirates,NaN,"Middle East, North Africa, Afghanistan & Pakistan",High income,Abu Dhabi,24.4764,54.3705,ARE,2024.0,1.663365,72.4,34.2,68.5,70.8,96.2,NaN,NaN,NaN,1100.0,NaN
10,Australia,NaN,East Asia & Pacific,High income,Canberra,-35.2820,149.1290,AUS,2024.0,3.166567,69.5,32.8,65.4,68.9,95.6,3.25,NaN,NaN,1450.0,140.0
17,Bangladesh,NaN,South Asia,Lower middle income,Dhaka,23.7055,90.4113,BGD,2024.0,10.465748,39.5,8.8,38.1,38.2,50.8,0.95,1.18,14.5,100.0,25.0
19,Bahrain,NaN,"Middle East, North Africa, Afghanistan & Pakistan",High income,Manama,26.1921,50.5354,BHR,2024.0,0.919635,65.2,26.1,61.8,63.5,86.2,NaN,NaN,NaN,600.0,NaN


### 3. Definição do 5-Fold Cross Validation
Configuramos o validador cruzado para dividir a base em 5 partes e embaralhar os dados.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

### 4. Modelo 1: Random Forest (com ajuste de parâmetros via GridSearchCV)

In [ ]:
# Definindo os hiperparâmetros para testar no GridSearch
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20]
}

rf = RandomForestClassifier(random_state=42)

# Buscando os melhores parâmetros (Tuning)
grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_scaled, y_encoded)

best_rf = grid_rf.best_estimator_
print("Melhores parâmetros para Random Forest:", grid_rf.best_params_)

# Avaliação final do melhor modelo usando 5-fold CV na base toda
cv_scores_rf = cross_val_score(best_rf, X_scaled, y_encoded, cv=kf, scoring='accuracy')
print(f"Acurácia média 5-fold CV (Random Forest): {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})")

Melhores parâmetros para Random Forest: {'max_depth': None, 'n_estimators': 50}
Acurácia média 5-fold CV (Random Forest): 0.7694 (+/- 0.1556)


### 5. Modelo 2: Support Vector Machine - SVM (com ajuste de parâmetros)

In [ ]:
# Definindo os parâmetros para testar no SVM
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

svm_base = SVC(random_state=42)

# Buscando os melhores parâmetros (Tuning)
grid_svm = GridSearchCV(svm_base, param_grid_svm, cv=5, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_scaled, y_encoded)

best_svm = grid_svm.best_estimator_
print("Melhores parâmetros para SVM:", grid_svm.best_params_)

# Avaliação final com 5-fold CV
cv_scores_svm = cross_val_score(best_svm, X_scaled, y_encoded, cv=kf, scoring='accuracy')
print(f"Acurácia média 5-fold CV (SVM): {cv_scores_svm.mean():.4f} (+/- {cv_scores_svm.std():.4f})")

Melhores parâmetros para SVM: {'C': 1, 'kernel': 'linear'}
Acurácia média 5-fold CV (SVM): 0.8611 (+/- 0.0819)


### 6. Modelo 3: K-Nearest Neighbors (KNN)
Para o KNN, manteremos os parâmetros padrões (n_neighbors=5) para cumprir exatamente o requisito da atividade (3 modelos no total, sendo 2 com parâmetros ajustados).

In [ ]:
knn = KNeighborsClassifier()

# Avaliação direta com 5-fold CV
cv_scores_knn = cross_val_score(knn, X_scaled, y_encoded, cv=kf, scoring='accuracy')
print(f"Acurácia média 5-fold CV (KNN): {cv_scores_knn.mean():.4f} (+/- {cv_scores_knn.std():.4f})")

Acurácia média 5-fold CV (KNN): 0.7444 (+/- 0.1334)


### 7. Comparação e Conclusão (Discussão dos Resultados)

In [ ]:
print("====== COMPARAÇÃO FINAL DE CLASSIFICADORES (5-FOLD CV) ======")
print(f"1. Random Forest (Tuning via GridSearch) : {cv_scores_rf.mean()*100:.2f}%")
print(f"2. SVM (Tuning via GridSearch)           : {cv_scores_svm.mean()*100:.2f}%")
print(f"3. KNN (Configuração Padrão)             : {cv_scores_knn.mean()*100:.2f}%")

# Escolhendo o melhor e imprimindo a explicação para a atividade
resultados = {'Random Forest': cv_scores_rf.mean(), 'SVM': cv_scores_svm.mean(), 'KNN': cv_scores_knn.mean()}
melhor_modelo = max(resultados, key=resultados.get)

print("\n--- Conclusão da Atividade ---")
print(f"O objetivo foi classificar a variável socioeconômica 'income_level' utilizando métricas de inflação e custo de vida.")
print(f"O melhor classificador criado foi o **{melhor_modelo}**, atingindo uma acurácia de {resultados[melhor_modelo]*100:.2f}% na validação cruzada (5-fold).")
print("Isso demonstra que o algoritmo conseguiu generalizar os padrões complexos dos índices econômicos, associando corretamente se o país era de alta renda, média ou baixa renda com uma margem de acerto satisfatória dadas as limitações da base.")

====== COMPARAÇÃO FINAL DE CLASSIFICADORES (5-FOLD CV) ======
1. Random Forest (Tuning via GridSearch) : 76.94%
2. SVM (Tuning via GridSearch)           : 86.11%
3. KNN (Configuração Padrão)             : 74.44%

--- Conclusão da Atividade ---
O objetivo foi classificar a variável socioeconômica 'income_level' utilizando métricas de inflação e custo de vida.
O melhor classificador criado foi o **SVM**, atingindo uma acurácia de 86.11% na validação cruzada (5-fold).
Isso demonstra que o algoritmo conseguiu generalizar os padrões complexos dos índices econômicos, associando corretamente se o país era de alta renda, média ou baixa renda com uma margem de acerto satisfatória dadas as limitações da base.
